In [2]:
from pathlib import Path

learn_project_root = Path.cwd()

learn_data_root = (
    learn_project_root
    / "data"
    / "raw"
    / "plantseg-v7"
    / "plantseg"
)

learn_metadata_path = learn_data_root / "Metadata.csv"

print("当前工作目录：", learn_project_root)
print("数据目录存在：", learn_data_root.is_dir())
print("Metadata.csv 存在：", learn_metadata_path.is_file())

当前工作目录： /Users/apple/Desktop/大四上/AIP/CW2
数据目录存在： True
Metadata.csv 存在： True


In [3]:
import pandas as pd

assert learn_metadata_path.is_file(), "找不到 Metadata.csv，请检查上一个单元格的路径"

learn_meta = pd.read_csv(
    learn_metadata_path,
    keep_default_na=False
)

print("行数和列数：", learn_meta.shape)
print("字段名称：", learn_meta.columns.tolist())

display(learn_meta.head(5))

行数和列数： (7774, 10)
字段名称： ['Name', 'Index', 'Plant', 'Disease', 'Resolution', 'Label file', 'Mask ratio', 'URL', 'License', 'Split']


,Name,Index,Plant,Disease,Resolution,Label file,Mask ratio,URL,License,Split
0,apple_black_rot_1.jpg,0,Apple,apple black rot,640x480,apple_black_rot_1.png,0.040439,https://baijiahao.baidu.com/s?id=1592172448991...,CC-BY-NC,Training
1,apple_black_rot_11.jpg,0,Apple,apple black rot,1080x755,apple_black_rot_11.png,0.057550,http://dy.163.com/v2/article/detail/EJ3OGOG605...,CC-BY-NC,Training
2,apple_black_rot_12.jpg,0,Apple,apple black rot,1280x719,apple_black_rot_12.png,0.024791,http://baijiahao.baidu.com/s?id=16205160930871...,CC-BY-NC,Training
3,apple_black_rot_15.jpg,0,Apple,apple black rot,600x355,apple_black_rot_15.png,0.236169,http://www.360doc.cn/article/4310958_180040780...,CC-BY-NC,Training
4,apple_black_rot_156.jpg,0,Apple,apple black rot,219x294,apple_black_rot_156.png,0.054344,http://baijiahao.baidu.com/s?id=16352401074244...,CC-BY-NC,Training


In [4]:
learn_columns = [
    "Name",
    "Plant",
    "Disease",
    "Index",
    "Label file",
    "Split"
]

display(learn_meta[learn_columns].head(8))

,Name,Plant,Disease,Index,Label file,Split
0,apple_black_rot_1.jpg,Apple,apple black rot,0,apple_black_rot_1.png,Training
1,apple_black_rot_11.jpg,Apple,apple black rot,0,apple_black_rot_11.png,Training
2,apple_black_rot_12.jpg,Apple,apple black rot,0,apple_black_rot_12.png,Training
3,apple_black_rot_15.jpg,Apple,apple black rot,0,apple_black_rot_15.png,Training
4,apple_black_rot_156.jpg,Apple,apple black rot,0,apple_black_rot_156.png,Training
5,apple_black_rot_187.jpg,Apple,apple black rot,0,apple_black_rot_187.png,Training
6,apple_black_rot_28.jpg,Apple,apple black rot,0,apple_black_rot_28.png,Validation
7,apple_black_rot_34.jpg,Apple,apple black rot,0,apple_black_rot_34.png,Training


In [5]:
learn_hosts = [
    "apple", "banana", "citrus",
    "corn", "potato", "soybean",
    "tomato", "wheat", "zucchini"
]

learn_meta["host"] = (
    learn_meta["Plant"]
    .str.strip()
    .str.lower()
)

learn_selected = learn_meta.loc[
    learn_meta["host"].isin(learn_hosts)
].copy()

print("amount of pics before selected: ", len(learn_meta))
print("amount of pics after selected: ", len(learn_selected))
print("amount of hosts after selected: ", learn_selected["host"].nunique())
print("actual hosts: ", sorted(learn_selected["host"].unique()))

assert set (learn_selected["host"]) == set(learn_hosts), "缺少某些 host，请检查数据"

amount of pics before selected:  7774
amount of pics after selected:  4349
amount of hosts after selected:  9
actual hosts:  ['apple', 'banana', 'citrus', 'corn', 'potato', 'soybean', 'tomato', 'wheat', 'zucchini']


In [6]:
display(
    learn_selected[["Plant", "host", "Disease"]].head(10)
)

,Plant,host,Disease
0,Apple,apple,apple black rot
1,Apple,apple,apple black rot
2,Apple,apple,apple black rot
3,Apple,apple,apple black rot
4,Apple,apple,apple black rot
5,Apple,apple,apple black rot
6,Apple,apple,apple black rot
7,Apple,apple,apple black rot
8,Apple,apple,apple black rot
9,Apple,apple,apple black rot


In [7]:
learn_diseases = []

for host, raw_label in zip(
    learn_selected["host"],
    learn_selected["Disease"]
):
    label = " ".join(raw_label.strip().lower().split())
    prefix = host + " "

    if not label.startswith(prefix):
        raise ValueError(f"疾病标签 '{label}' 不以宿主 '{host}' 开头，请检查数据")

    disease = label[len(prefix):]
    learn_diseases.append(disease)

learn_selected["disease"] = learn_diseases

display(
    learn_selected[["Plant", "host", "Disease", "disease"]].drop_duplicates().head(10)
)

,Plant,host,Disease,disease
0,Apple,apple,apple black rot,black rot
51,Apple,apple,apple mosaic virus,mosaic virus
111,Apple,apple,apple rust,rust
187,Apple,apple,apple scab,scab
320,Banana,banana,banana anthracnose,anthracnose
361,Banana,banana,banana black leaf streak,black leaf streak
450,Banana,banana,banana bunchy top,bunchy top
515,Banana,banana,banana cigar end rot,cigar end rot
539,Banana,banana,banana cordana leaf spot,cordana leaf spot
561,Banana,banana,banana panama disease,panama disease


In [9]:
learn_classes=(
    learn_selected[["host", "disease"]]
    .drop_duplicates()
    .sort_values(["host", "disease"])
    .reset_index(drop=True)
)

print("amount of classes: ", len(learn_classes))

display(learn_classes)

amount of classes:  43


,host,disease
0,apple,black rot
1,apple,mosaic virus
2,apple,rust
3,apple,scab
4,banana,anthracnose
5,banana,black leaf streak
6,banana,bunchy top
7,banana,cigar end rot
8,banana,cordana leaf spot
9,banana,panama disease


In [10]:
learn_classes_per_host = (
    learn_classes.groupby("host")
    .size()
    .rename("class_count")
)

display(learn_classes_per_host)

print("所有宿主的类别数之和：", learn_classes_per_host.sum())

host
apple       4
banana      6
citrus      2
corn        4
potato      2
soybean     6
tomato      7
wheat       8
zucchini    4
Name: class_count, dtype: int64

所有宿主的类别数之和： 43


In [18]:
print("原始分区名称：", learn_selected["Split"].unique())

learn_split_counts = (
    learn_selected["Split"]
    .value_counts()
    .reindex(["Training", "Validation", "Test"], fill_value=0)
)

display(learn_split_counts)

learn_split_percent = (
    learn_split_counts / learn_split_counts.sum() * 100
).round(2)

display(learn_split_percent.rename("percent"))

learn_host_counts = (
    learn_selected
    .groupby(["host", "Split"])
    .size()
    .unstack("Split", fill_value=0)
    .reindex(
        index=learn_hosts,
        columns=["Training", "Validation", "Test"],
        fill_value=0
    )
)

display(learn_host_counts)

learn_class_counts = (
    learn_selected
    .groupby(["host", "disease", "Split"])
    .size()
    .unstack("Split", fill_value=0)
    .reindex(
        columns=["Training", "Validation", "Test"],
        fill_value=0
    )
)

learn_class_counts["Total"] = (
    learn_class_counts[
        ["Training", "Validation", "Test"]
    ].sum(axis=1)
)

display(learn_class_counts)

原始分区名称： <StringArray>
['Training', 'Validation', 'Test']
Length: 3, dtype: str


Split
Training      3007
Validation     475
Test           867
Name: count, dtype: int64

Split
Training      69.14
Validation    10.92
Test          19.94
Name: percent, dtype: float64

Split,Training,Validation,Test
host,,,
apple,274,46,82
banana,244,36,74
citrus,291,42,83
corn,289,46,84
potato,103,16,27
soybean,356,46,98
tomato,460,74,133
wheat,798,137,231
zucchini,192,32,55


Split                                         Training  Validation  Test  \
host     disease                                                           
apple    black rot                                  43           8    12   
         mosaic virus                               52           8    13   
         rust                                       64          12    22   
         scab                                      115          18    35   
banana   anthracnose                                35           6     9   
         black leaf streak                          79          10    25   
         bunchy top                                 56           9    13   
         cigar end rot                              22           2     9   
         cordana leaf spot                          19           3     8   
         panama disease                             33           6    10   
citrus   canker                                    225          33    65   
         greening disease                           66           9    18   
corn     gray leaf spot                             49          10    17   
         northern leaf blight                       65           9    18   
         rust                                       84          13    23   
         smut                                       91          14    26   
potato   early blight                               50           9     9   
         late blight                                53           7    18   
soybean  bacterial blight                           41           6    14   
         brown spot                                 42           3    10   
         downy mildew                               56           6    14   
         frog eye leaf spot                        111          13    29   
         mosaic                                     61          11    17   
         rust                                       45           7    14   
tomato   bacterial leaf spot                        64           9    17   
         early blight                              103          16    34   
         late blight                                84          14    19   
         leaf mold                                  71          13    18   
         mosaic virus                               29           5     9   
         septoria leaf spot                         66          13    20   
         yellow leaf curl virus                     43           4    16   
wheat    bacterial leaf streak (black chaff)        53          11    19   
         head scab                                 161          25    44   
         leaf rust                                  53           7    15   
         loose smut                                101          15    25   
         powdery mildew                            119          24    35   
         septoria blotch                            95          19    28   
         stem rust                                  56          11    22   
         stripe rust                               160          25    43   
zucchini bacterial wilt                             38           7    12   
         downy mildew                               25           3     5   
         powdery mildew                             90          16    27   
         yellow mosaic virus                        39           6    11   

Split                                         Total  
host     disease                                     
apple    black rot                               63  
         mosaic virus                            73  
         rust                                    98  
         scab                                   168  
banana   anthracnose                             50  
         black leaf streak                      114  
         bunchy top                              78  
         cigar end rot                           33  
         cordana leaf spot             

In [20]:
assert len(learn_class_counts) == len(learn_classes), "类别数量不匹配，请检查数据"

assert (
    learn_class_counts["Total"].sum()
    == len(learn_selected)
)

assert (
    learn_class_counts[
        ["Training", "Validation", "Test"]
    ].sum(axis=0)
    .eq(learn_split_counts)
    .all()
)

print("检查通过：类别数量、图片总数和各分区数量一致")

检查通过：类别数量、图片总数和各分区数量一致


In [21]:
learn_zero_partition_classes = learn_class_counts.loc[
    learn_class_counts[
        ["Training", "Validation", "Test"]
    ].eq(0).any(axis=1)
]

print("至少一个原始分区为零的类别数：",
      len(learn_zero_partition_classes))

display(learn_zero_partition_classes)

至少一个原始分区为零的类别数： 0


,Split,Training,Validation,Test,Total
host,disease,,,,


In [ ]:
#assignment A
learn_roles_A = (
    learn_classes
    .sort_values(["host", "disease"])
    .reset_index(drop=True)
    .copy()
)

learn_roles_A["disease_rank"] = (
    learn_roles_A.groupby("host").cumcount() + 1
)

display(learn_roles_A)

,host,disease,disease_rank
0,apple,black rot,1
1,apple,mosaic virus,2
2,apple,rust,3
3,apple,scab,4
4,banana,anthracnose,1
5,banana,black leaf streak,2
6,banana,bunchy top,3
7,banana,cigar end rot,4
8,banana,cordana leaf spot,5
9,banana,panama disease,6


In [23]:
learn_roles_A["class_role"] = "known"

learn_familiar_host = ~learn_roles_A["host"].isin(["citrus", "potato"])

learn_roles_A.loc[
    learn_familiar_host & learn_roles_A["disease_rank"].eq(1),
    "class_role"
] = "development_unknown"

learn_roles_A.loc[
    learn_familiar_host
    & learn_roles_A["disease_rank"].eq(2),
    "class_role"
] = "test_unknown"

learn_roles_A.loc[
    learn_roles_A["host"].eq("citrus"),
    "class_role"
] = "development_unknown"

learn_roles_A.loc[
    learn_roles_A["host"].eq("potato"),
    "class_role"
] = "test_unknown"

learn_apple_A = learn_roles_A.loc[
    learn_roles_A["host"].eq("apple"),
    ["host", "disease", "class_role"]
]

display(learn_apple_A)

,host,disease,class_role
0,apple,black rot,development_unknown
1,apple,mosaic virus,test_unknown
2,apple,rust,known
3,apple,scab,known


In [24]:
learn_expected_apple_A = {
    "black rot": "development_unknown",
    "mosaic virus": "test_unknown",
    "rust": "known",
    "scab": "known"
}

learn_actual_apple_A = (
    learn_apple_A
    .set_index("disease")["class_role"]
    .to_dict()
)

assert learn_actual_apple_A == learn_expected_apple_A

assert learn_roles_A.loc[
    learn_roles_A["host"].eq("citrus"),
    "class_role"
].eq("development_unknown").all()

assert learn_roles_A.loc[
    learn_roles_A["host"].eq("potato"),
    "class_role"
].eq("test_unknown").all()

assert not learn_roles_A.duplicated(
    ["host", "disease"]
).any()

assert len(learn_roles_A) == len(learn_classes)

display(learn_roles_A["class_role"].value_counts())

class_role
known                  25
development_unknown     9
test_unknown            9
Name: count, dtype: int64

In [26]:
#assignment B
learn_roles_B = learn_roles_A[
    ["host", "disease", "disease_rank"]
].copy()

learn_roles_B["class_role"] = "known"

learn_familiar_host_B = ~learn_roles_B["host"].isin(
    ["citrus", "potato"]
)

learn_roles_B.loc[
    learn_familiar_host_B
    & learn_roles_B["disease_rank"].eq(2),
    "class_role"
] = "development_unknown"

learn_roles_B.loc[
    learn_familiar_host_B
    & learn_roles_B["disease_rank"].eq(3),
    "class_role"
] = "test_unknown"

learn_roles_B.loc[
    learn_roles_B["host"].eq("citrus"),
    "class_role"
] = "development_unknown"

learn_roles_B.loc[
    learn_roles_B["host"].eq("potato"),
    "class_role"
] = "test_unknown"

display(learn_roles_B.loc[
    learn_roles_B["host"].eq("apple")
])

,host,disease,disease_rank,class_role
0,apple,black rot,1,known
1,apple,mosaic virus,2,development_unknown
2,apple,rust,3,test_unknown
3,apple,scab,4,known


In [31]:
learn_expected_apple_B = {
    "black rot": "known",
    "mosaic virus": "development_unknown",
    "rust": "test_unknown",
    "scab": "known"
}

learn_actual_apple_B = (
    learn_roles_B.loc[learn_roles_B["host"].eq("apple")]
    .set_index("disease")["class_role"]
    .to_dict()
)

assert learn_actual_apple_B == learn_expected_apple_B

assert learn_roles_B.loc[
    learn_roles_B["host"].eq("citrus"),
    "class_role"
].eq("development_unknown").all()

assert learn_roles_B.loc[
    learn_roles_B["host"].eq("potato"),
    "class_role"
].eq("test_unknown").all()

display(learn_roles_B["class_role"].value_counts())

class_role
known                  25
development_unknown     9
test_unknown            9
Name: count, dtype: int64

In [32]:
learn_role_comparison = learn_roles_A[
    ["host", "disease", "class_role"]
].merge(
    learn_roles_B[["host", "disease", "class_role"]],
    on=["host", "disease"],
    how="outer",
    suffixes=("_A", "_B"),
    validate="one_to_one"
)

assert len(learn_role_comparison) == len(learn_classes)

assert learn_role_comparison[
    ["class_role_A", "class_role_B"]
].notna().all().all()

display(learn_role_comparison.loc[
    learn_role_comparison["host"].eq("apple")
])

,host,disease,class_role_A,class_role_B
0,apple,black rot,development_unknown,known
1,apple,mosaic virus,test_unknown,development_unknown
2,apple,rust,known,test_unknown
3,apple,scab,known,known


In [33]:
learn_changed_roles = learn_role_comparison.loc[
    learn_role_comparison["class_role_A"].ne(
        learn_role_comparison["class_role_B"]
    )
]

print("A/B 角色发生变化的类别数：", len(learn_changed_roles))

display(learn_changed_roles)

A/B 角色发生变化的类别数： 21


,host,disease,class_role_A,class_role_B
0,apple,black rot,development_unknown,known
1,apple,mosaic virus,test_unknown,development_unknown
2,apple,rust,known,test_unknown
4,banana,anthracnose,development_unknown,known
5,banana,black leaf streak,test_unknown,development_unknown
6,banana,bunchy top,known,test_unknown
12,corn,gray leaf spot,development_unknown,known
13,corn,northern leaf blight,test_unknown,development_unknown
14,corn,rust,known,test_unknown
18,soybean,bacterial blight,development_unknown,known


In [35]:
manifest_A = learn_selected.merge(
    learn_roles_A[
        ["host", "disease", "class_role"]
    ],
    on=["host", "disease"],
    how="left",
    validate="many_to_one"
)

assert len(manifest_A) == len(learn_selected), "manifest_A 的行数与 learn_selected 不匹配"
assert manifest_A["class_role"].notna().all(), "manifest_A 中存在缺失的 class_role"
assert manifest_A["Name"].is_unique

display(
    manifest_A[
        ["Name", "host", "disease", "Split", "class_role"]
    ].head(10)
)

,Name,host,disease,Split,class_role
0,apple_black_rot_1.jpg,apple,black rot,Training,development_unknown
1,apple_black_rot_11.jpg,apple,black rot,Training,development_unknown
2,apple_black_rot_12.jpg,apple,black rot,Training,development_unknown
3,apple_black_rot_15.jpg,apple,black rot,Training,development_unknown
4,apple_black_rot_156.jpg,apple,black rot,Training,development_unknown
5,apple_black_rot_187.jpg,apple,black rot,Training,development_unknown
6,apple_black_rot_28.jpg,apple,black rot,Validation,development_unknown
7,apple_black_rot_34.jpg,apple,black rot,Training,development_unknown
8,apple_black_rot_38.jpg,apple,black rot,Training,development_unknown
9,apple_black_rot_4.jpg,apple,black rot,Training,development_unknown


In [39]:
manifest_A["assignment"] = "A"
manifest_A["experimental_split"] = "unused"

known_A = manifest_A["class_role"].eq("known")

split_names = {
    "Training": "train",
    "Validation": "validation",
    "Test": "test"
}

manifest_A.loc[
    known_A,
    "experimental_split"
] = (
    manifest_A.loc[known_A, "Split"]
    .map(split_names)
)

dev_validation_A = (
    manifest_A["class_role"].eq("development_unknown")
    & manifest_A["Split"].eq("Validation")
)

manifest_A.loc[
    dev_validation_A,
    "experimental_split"
] = "validation"

unknown_test_A = (
    manifest_A["class_role"].eq("test_unknown")
    & manifest_A["Split"].eq("Test")
)

manifest_A.loc[
    unknown_test_A,
    "experimental_split"
] = "test"

black_rot_A = (
    manifest_A["host"].eq("apple")
    & manifest_A["disease"].eq("black rot")
)

display(
    manifest_A[black_rot_A]
    .groupby(["Split", "experimental_split"])
    .size()
    .rename("image_count")
)

Split       experimental_split
Test        unused                12
Training    unused                43
Validation  validation             8
Name: image_count, dtype: int64

In [40]:
def build_manifest(image, roles, assignment):
    manifest = image.merge(
        roles[["host", "disease", "class_role"]],
        on=["host", "disease"],
        how="left",
        validate="many_to_one"
    )

    assert len(manifest) == len(image), "manifest 的行数与 image 不匹配"
    assert manifest["Name"].is_unique
    assert manifest["class_role"].notna().all()

    manifest["assignment"] = assignment
    manifest["experimental_split"] = "unused"

    known = manifest["class_role"].eq("known")

    split_names = {
        "Training": "train",
        "Validation": "validation",
        "Test": "test"
    }

    manifest.loc[
        known,
        "experimental_split"
    ] = (
        manifest.loc[known, "Split"]
        .map(split_names)
    )

    dev_validation = (
        manifest["class_role"].eq("development_unknown")
        & manifest["Split"].eq("Validation")
    )

    manifest.loc[
        dev_validation,
        "experimental_split"
    ] = "validation"

    unknown_test = (
        manifest["class_role"].eq("test_unknown")
        & manifest["Split"].eq("Test")
    )

    manifest.loc[
        unknown_test,
        "experimental_split"
    ] = "test"

    return manifest

In [42]:
manifest_A = build_manifest(
    learn_selected,
    learn_roles_A,
    "A"
)

manifest_B = build_manifest(
    learn_selected,
    learn_roles_B,
    "B"
)

print("A 的图片记录数：", len(manifest_A))
print("B 的图片记录数：", len(manifest_B))

A 的图片记录数： 4349
B 的图片记录数： 4349


In [44]:
for assignment, manifest in [
    ("A", manifest_A),
    ("B", manifest_B)
]:
    black_rot = manifest.loc[
        manifest["host"].eq("apple")
        & manifest["disease"].eq("black rot")
    ]

    print("Assignment", assignment)

    display(
        black_rot.groupby(
            ["class_role", "Split", "experimental_split"]
        ).size().rename("image_count")
    )

Assignment A


class_role           Split       experimental_split
development_unknown  Test        unused                12
                     Training    unused                43
                     Validation  validation             8
Name: image_count, dtype: int64

Assignment B


class_role  Split       experimental_split
known       Test        test                  12
            Training    train                 43
            Validation  validation             8
Name: image_count, dtype: int64

In [46]:
def check_manifest(manifest, original_images):
    assert len(manifest) == len(original_images)
    assert manifest["Name"].is_unique
    assert manifest["experimental_split"].notna().all()

    assert set(manifest["experimental_split"]).issubset(
        {"train", "validation", "test", "unused"}
    )

    original_splits = (
        original_images.set_index("Name")["Split"].sort_index()
    )

    current_splits = (
        manifest.set_index("Name")["Split"].sort_index()
    )

    assert current_splits.equals(original_splits)

    train = manifest.loc[
        manifest["experimental_split"].eq("train")
    ]

    assert train["class_role"].eq("known").all()
    assert train["Split"].eq("Training").all()

    used_dev = manifest.loc[
        manifest["class_role"].eq("development_unknown")
        & manifest["experimental_split"].ne("unused")
    ]

    assert used_dev["Split"].eq("Validation").all()
    assert used_dev["experimental_split"].eq("validation").all()

    used_test = manifest.loc[
        manifest["class_role"].eq("test_unknown")
        & manifest["experimental_split"].ne("unused")
    ]

    assert used_test["Split"].eq("Test").all()
    assert used_test["experimental_split"].eq("test").all()

In [48]:
check_manifest(manifest_A, learn_selected)
check_manifest(manifest_B, learn_selected)

print("A/B manifest 协议检查通过")

A/B manifest 协议检查通过


In [49]:
for assignment, manifest in [
    ("A", manifest_A),
    ("B", manifest_B)
]:
    print("Assignment", assignment)

    display(
        manifest.groupby(
            ["experimental_split", "class_role"]
        ).size().rename("image_count")
    )

Assignment A


experimental_split  class_role         
test                known                   508
                    test_unknown            176
train               known                  1763
unused              development_unknown     797
                    test_unknown            720
validation          development_unknown      99
                    known                   286
Name: image_count, dtype: int64

Assignment B


experimental_split  class_role         
test                known                   475
                    test_unknown            160
train               known                  1599
unused              development_unknown    1050
                    test_unknown            683
validation          development_unknown     116
                    known                   266
Name: image_count, dtype: int64

In [50]:
def add_known_ids(manifest, roles):
    known_classes = (
        roles.loc[
            roles["class_role"].eq("known"),
            ["host", "disease"]
        ]
        .sort_values(["host", "disease"])
        .reset_index(drop=True)
        .copy()
    )

    known_classes["known_class_id"] = pd.array(
        range(len(known_classes)),
        dtype="Int64"
    )

    result = manifest.drop(
        columns=["known_class_id"], errors="ignore"
    ).merge(
        known_classes,
        on=["host", "disease"],
        how="left",
        validate="many_to_one"
    )

    known = result["class_role"].eq("known")

    assert result.loc[known, "known_class_id"].notna().all()

    assert result.loc[~known, "known_class_id"].isna().all()

    return result, known_classes

In [51]:
manifest_A, known_ids_A = add_known_ids(
    manifest_A,
    learn_roles_A
)

manifest_B, known_ids_B = add_known_ids(
    manifest_B,
    learn_roles_B
)

display(known_ids_A.head(10))
display(known_ids_B.head(10))

,host,disease,known_class_id
0,apple,rust,0
1,apple,scab,1
2,banana,bunchy top,2
3,banana,cigar end rot,3
4,banana,cordana leaf spot,4
5,banana,panama disease,5
6,corn,rust,6
7,corn,smut,7
8,soybean,downy mildew,8
9,soybean,frog eye leaf spot,9


,host,disease,known_class_id
0,apple,black rot,0
1,apple,scab,1
2,banana,anthracnose,2
3,banana,cigar end rot,3
4,banana,cordana leaf spot,4
5,banana,panama disease,5
6,corn,gray leaf spot,6
7,corn,smut,7
8,soybean,bacterial blight,8
9,soybean,frog eye leaf spot,9


In [53]:
for assignment, known_ids in [
    ("A", known_ids_A),
    ("B", known_ids_B)
]:
    actual_ids = known_ids["known_class_id"].tolist()
    expected_ids = list(range(len(known_ids)))

    assert actual_ids == expected_ids

    assert not known_ids.duplicated(
        ["host", "disease"]
    ).any()

    print(
        assignment,
        "known 类别数：",
        len(known_ids),
        "编号范围：",
        actual_ids[0],
        "到",
        actual_ids[-1]
    )

A known 类别数： 25 编号范围： 0 到 24
B known 类别数： 25 编号范围： 0 到 24


In [54]:
from pathlib import PurePosixPath

def add_paths(manifest):
    result = manifest.copy()

    folder_names = {
        "Training": "train",
        "Validation": "validation",
        "Test": "test"
    }

    folders = result["Split"].map(folder_names)

    assert folders.notna().all()

    result["image_id"] = result["Name"]

    result["image_path"] = [
        str(PurePosixPath("images") / folder / name)
        for folder, name in zip(folders, result["Name"])
    ]

    result["mask_path"] = [
        str(PurePosixPath("annotations") / folder / name)
        for folder, name in zip(folders, result["Label file"])
    ]

    return result

In [55]:
manifest_A = add_paths(manifest_A)
manifest_B = add_paths(manifest_B)

display(
    manifest_A[
        [
            "image_id",
            "Split",
            "experimental_split",
            "known_class_id",
            "image_path",
            "mask_path"
        ]
    ].head(8)
)

,image_id,Split,experimental_split,known_class_id,image_path,mask_path
0,apple_black_rot_1.jpg,Training,unused,<NA>,images/train/apple_black_rot_1.jpg,annotations/train/apple_black_rot_1.png
1,apple_black_rot_11.jpg,Training,unused,<NA>,images/train/apple_black_rot_11.jpg,annotations/train/apple_black_rot_11.png
2,apple_black_rot_12.jpg,Training,unused,<NA>,images/train/apple_black_rot_12.jpg,annotations/train/apple_black_rot_12.png
3,apple_black_rot_15.jpg,Training,unused,<NA>,images/train/apple_black_rot_15.jpg,annotations/train/apple_black_rot_15.png
4,apple_black_rot_156.jpg,Training,unused,<NA>,images/train/apple_black_rot_156.jpg,annotations/train/apple_black_rot_156.png
5,apple_black_rot_187.jpg,Training,unused,<NA>,images/train/apple_black_rot_187.jpg,annotations/train/apple_black_rot_187.png
6,apple_black_rot_28.jpg,Validation,validation,<NA>,images/validation/apple_black_rot_28.jpg,annotations/validation/apple_black_rot_28.png
7,apple_black_rot_34.jpg,Training,unused,<NA>,images/train/apple_black_rot_34.jpg,annotations/train/apple_black_rot_34.png


In [56]:
example = manifest_A.iloc[0]

image_file = (
    learn_data_root / example["image_path"]
)

mask_file = (
    learn_data_root / example["mask_path"]
)

print("图片存在：", image_file.is_file())
print("Mask 存在：", mask_file.is_file())

图片存在： True
Mask 存在： True
